# 🥁 VS-Batera no Google Colab (GPU)

Processa **todas as músicas** de uma pasta do seu Google Drive usando a GPU do Colab:
separa com **Demucs + RoFormer** (combinados), tira o resto de bateria numa **2ª passada**,
limpa cada faixa e gera a **versão sem bateria**. Na GPU leva poucos minutos por música
(no processador do PC são 40–90 min).

**Antes de começar**
1. Menu **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU** (T4, L4 ou A100).
2. No seu Google Drive, crie a pasta **`VS-Batera/musicas`** e coloque as músicas lá (mp3, wav, flac, m4a).
3. Rode as células em ordem (▶ em cada uma, ou **Ambiente de execução → Executar tudo**).

Os resultados vão para **`VS-Batera/saida/<música>/`** no Drive. Músicas já processadas são puladas,
então se o Colab desconectar é só rodar de novo que ele continua de onde parou.

In [ ]:
#@title 1. Conectar o Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

PASTA_DRIVE = '/content/drive/MyDrive/VS-Batera'  #@param {type:"string"}
MUSICAS = Path(PASTA_DRIVE) / 'musicas'
SAIDA = Path(PASTA_DRIVE) / 'saida'
MODELOS = Path(PASTA_DRIVE) / '.modelos'   # os modelos ficam no Drive: só baixa uma vez
for p in (MUSICAS, SAIDA, MODELOS):
    p.mkdir(parents=True, exist_ok=True)

# músicas enviadas em pedaços (nome.mp3.parte00, .parte01, ...): junta de volta, byte a byte
import re, collections
pedacos = collections.defaultdict(list)
for p in MUSICAS.iterdir():
    m = re.match(r'(.+)\.parte(\d+)$', p.name)
    if m:
        pedacos[m.group(1)].append((int(m.group(2)), p))
for nome, partes in pedacos.items():
    partes.sort()
    with open(MUSICAS / nome, 'wb') as saida:
        for _, parte in partes:
            saida.write(parte.read_bytes())
    for _, parte in partes:
        parte.unlink()
    print('juntou', nome, f'({len(partes)} pedaços)')

exts = {'.mp3', '.wav', '.flac', '.m4a', '.ogg', '.aac'}
lista = sorted(p for p in MUSICAS.iterdir() if p.suffix.lower() in exts)
print(f'{len(lista)} música(s) em {MUSICAS}')
for p in lista[:30]:
    print('  -', p.name)

In [ ]:
#@title 2. Conferir a GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "SEM GPU: troque o tipo de ambiente para GPU"
import torch
print('PyTorch', torch.__version__, '| GPU disponível:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Ative a GPU: Ambiente de execução → Alterar o tipo de ambiente de execução → GPU'

In [ ]:
#@title 3. Instalar o VS-Batera (1–3 min)
REPO = 'https://github.com/guimedeirostr/VS-Batera.git'  #@param {type:"string"}
BRANCH = 'main'  #@param {type:"string"}
# se houver VS-Batera/codigo.zip no Drive, instala dele (serve para repositório privado); senão clona do GitHub
import zipfile
ZIP = Path(PASTA_DRIVE) / 'codigo.zip'
!rm -rf /content/VS-Batera
if ZIP.exists():
    zipfile.ZipFile(ZIP).extractall('/content')
    print('Código vindo do Drive:', ZIP)
else:
    !GIT_TERMINAL_PROMPT=0 git clone -q --depth 1 -b {BRANCH} {REPO} /content/VS-Batera
# o PyTorch com CUDA já vem no Colab; não deixa o pip trocar por outro
!pip install -q -e /content/VS-Batera "audio-separator[gpu]" 2>&1 | grep -v "already satisfied" | tail -n 5
# o pip install -e só vale para sessões novas do Python; garante o import já nesta sessão
import sys
if '/content/VS-Batera' not in sys.path:
    sys.path.insert(0, '/content/VS-Batera')
import vsbatera, demucs, audio_separator
print('Instalado.')

In [ ]:
#@title 4. Configuração
#@markdown **Separação dupla** = Demucs + RoFormer combinados (mais limpo). Desmarque para usar só o Demucs.
SEPARACAO_DUPLA = True  #@param {type:"boolean"}
#@markdown **2ª passada** = separa de novo a versão sem bateria para tirar o resto de prato/caixa.
SEGUNDA_PASSADA = True  #@param {type:"boolean"}
#@markdown **Calibração automática** da limpeza de cada faixa (a partir da análise).
CALIBRACAO_AUTO = True  #@param {type:"boolean"}
#@markdown **Guardar "só Demucs" e "só RoFormer"** para comparar no painel (ocupa ~2x mais espaço no Drive).
GUARDAR_VERSOES = False  #@param {type:"boolean"}
#@markdown Formato das faixas: **flac** é sem perda e ocupa ~metade do wav.
FORMATO_FAIXAS = 'flac'  #@param ["flac", "wav"]
#@markdown Formato da versão final sem bateria:
FORMATO_FINAL = 'wav'  #@param ["wav", "mp3", "flac"]
#@markdown Passadas extras do Demucs (mais = um pouco melhor e mais lento):
SHIFTS = 2  #@param {type:"slider", min:0, max:5, step:1}
#@markdown Reprocessar músicas que já têm resultado no Drive:
REPROCESSAR = False  #@param {type:"boolean"}

os.environ['VSBATERA_MODELOS'] = str(MODELOS)
os.environ['VSBATERA_FORMATO_FAIXAS'] = FORMATO_FAIXAS
print('ok')

In [ ]:
#@title 5. Processar as músicas (e ficar esperando novas)
#@markdown Com **VIGIAR** ligado, depois de processar tudo a célula continua rodando e processa
#@markdown cada música nova que aparecer em `VS-Batera/musicas` (por exemplo, as que o painel do PC manda)
#@markdown e atende os pedidos de **✨ Reconstruir** feitos no mixer do painel.
VIGIAR = True  #@param {type:"boolean"}
import glob, json, re, collections, shutil, time, traceback
from vsbatera import config as cfgmod, pipeline as pl
from vsbatera.audio_io import achar_faixas

TRABALHO = Path('/content/trabalho')   # processa no disco do Colab e só copia o resultado pronto pro Drive
STATUS = SAIDA / '_colab_status.json'  # o painel do PC lê este arquivo para mostrar o andamento


def gravar_status(**campos):
    try:
        STATUS.write_text(json.dumps({'atualizado': time.time(), **campos}, ensure_ascii=False), encoding='utf-8')
    except Exception:
        pass


def juntar_pedacos():
    pedacos = collections.defaultdict(list)
    for p in MUSICAS.iterdir():
        m = re.match(r'(.+)\.parte(\d+)$', p.name)
        if m:
            pedacos[m.group(1)].append((int(m.group(2)), p))
    for nome, partes in pedacos.items():
        partes.sort()
        with open(MUSICAS / nome, 'wb') as saida:
            for _, parte in partes:
                saida.write(parte.read_bytes())
        for _, parte in partes:
            parte.unlink()


def ja_processada(musica):
    proj = SAIDA / musica.stem
    return bool(achar_faixas(proj / 'limpas')) and any(proj.glob(glob.escape(musica.stem) + ' - sem bateria.*'))


def processar(musica, pendentes):
    preset = cfgmod.carregar_preset(None)
    s = preset['separacao']
    s['backend'] = 'duplo' if SEPARACAO_DUPLA else 'demucs'
    s['segunda_passada'] = SEGUNDA_PASSADA
    s['guardar_versoes'] = GUARDAR_VERSOES
    s['shifts'] = SHIFTS
    s['dispositivo'] = 'cuda'
    preset['mix']['formato'] = FORMATO_FINAL

    proj = TRABALHO / musica.stem
    shutil.rmtree(proj, ignore_errors=True)
    marco = [-1]

    def andamento(pct, etapa):
        gravar_status(processando=musica.stem, pct=pct, etapa=etapa, fila=pendentes)

    def progresso(f):
        pct = int(f * 100)
        if pct >= marco[0] + 10:
            marco[0] = pct - pct % 10
            print(f'    separando… {pct}%', flush=True)
            andamento(int(f * 80), 'separando')

    andamento(0, 'separando')
    pl.etapa_separar(musica, proj, preset, progresso=progresso)
    andamento(82, 'analisando')
    pl.etapa_analisar(proj, preset)
    if CALIBRACAO_AUTO:
        sugerido = cfgmod.carregar_preset(proj / 'preset_sugerido.yaml')
        sugerido['mix'] = preset['mix']
        preset = sugerido
    andamento(85, 'limpando')
    pl.etapa_limpar(proj, preset)
    andamento(95, 'mixando')
    pl.etapa_mixar(proj, preset)

    andamento(98, 'copiando para o Drive')
    destino = SAIDA / musica.stem
    shutil.rmtree(destino, ignore_errors=True)
    shutil.copytree(proj, destino)
    shutil.rmtree(proj, ignore_errors=True)
    return destino


def achar_projeto(nome):
    # o painel (Windows) escreve os nomes com acentos compostos (NFC); o Drive no Colab pode listar decompostos (NFD)
    import unicodedata
    alvo = unicodedata.normalize('NFC', nome)
    for d in SAIDA.iterdir():
        if d.is_dir() and unicodedata.normalize('NFC', d.name) == alvo:
            return d
    return None


def processar_pedidos():
    # pedidos de reconstrução (Apollo) feitos pelo botão '✨ Reconstruir' do painel
    pasta = SAIDA / '_pedidos'
    if not pasta.exists():
        return 0
    from vsbatera.reconstrucao import reconstruir_faixa
    feitos = 0
    for arq in sorted(pasta.glob('*.json')):
        try:
            d = json.loads(arq.read_text(encoding='utf-8'))
            proj, faixa = achar_projeto(d['projeto']), d['faixa']
            if proj is None:
                raise FileNotFoundError(d['projeto'])
            chave = d['projeto'] + '/' + faixa
            print(f'[reconstruir] {d["projeto"]} — {faixa}', flush=True)
            prog = lambda f: gravar_status(processando=None, reconstruindo=chave, pct=int(f * 100),
                                           etapa='reconstruindo ' + faixa, fila=[])
            prog(0)
            t0 = time.time()
            reconstruir_faixa(proj, faixa, 'cuda', progresso=prog)
            feitos += 1
            print(f'    ✅ reconstruída em {time.time() - t0:.0f} s', flush=True)
        except Exception:
            traceback.print_exc()
            print('    ⚠️ a reconstrução falhou:', arq.name, flush=True)
        finally:
            arq.unlink(missing_ok=True)
    return feitos


inicio_geral, feitas, erros, falharam, avisou = time.time(), 0, [], set(), False
while True:
    processar_pedidos()
    juntar_pedacos()
    musicas = sorted(p for p in MUSICAS.iterdir() if p.suffix.lower() in exts)
    pendentes = [m for m in musicas if m.name not in falharam and (REPROCESSAR or not ja_processada(m))]
    REPROCESSAR = False  # reprocessar vale só para a primeira rodada
    if pendentes:
        avisou = False
        print(f'{len(pendentes)} música(s) para processar\n')
    for i, musica in enumerate(pendentes, 1):
        processar_pedidos()  # reconstrução pedida no painel não espera o lote inteiro
        t = time.time()
        print(f'[{i}/{len(pendentes)}] {musica.stem}')
        try:
            destino = processar(musica, [m.stem for m in pendentes[i:]])
            feitas += 1
            print(f'    ✅ pronta em {(time.time() - t) / 60:.1f} min → {destino}\n')
        except Exception as e:
            falharam.add(musica.name)
            erros.append((musica.name, repr(e)))
            traceback.print_exc()
            print(f'    ⚠️ erro, pulando para a próxima\n')
    gravar_status(processando=None, pct=0, etapa='esperando músicas novas', fila=[], feitas=feitas,
                  erros=[n for n, _ in erros])
    if not VIGIAR:
        break
    if not avisou:
        print(f'Tudo em dia ({feitas} feita(s) nesta sessão). Esperando músicas novas em {MUSICAS}…')
        avisou = True
    time.sleep(20)

print(f'Fim: {feitas} pronta(s) em {(time.time() - inicio_geral) / 60:.0f} min.')
for nome, erro in erros:
    print('  ⚠️', nome, '→', erro)

## Ouvir e mixar no seu PC

Os resultados ficam no Drive em `VS-Batera/saida`. Para abrir no painel do VS-Batera:

1. Instale o **Google Drive para computador** (ele cria uma unidade, normalmente `G:`).
2. No PC, na pasta do projeto, rode o painel apontando para as pastas do Drive:

```bash
.venv\Scripts\vsbatera web --musicas "G:\Meu Drive\VS-Batera\musicas" -o "G:\Meu Drive\VS-Batera\saida"
```

Ou baixe só a pasta da música (`saida/<música>`) e coloque em `saida/` do projeto no PC.

**Dicas**
- O Colab grátis desconecta depois de um tempo; no **Colab Pro** a sessão dura mais. Se cair,
  rode as células de novo: as músicas prontas são puladas.
- Cada música ocupa uns 300–400 MB no Drive com FLAC e sem guardar as versões intermediárias.